# Titanic Classification
탑승객 데이터를 사용하여 탑승객의 생존 여부(Survived)를 분류
> 1차 테스트에서는 RandomForest 분류기가 정확도 약 0.83으로 가장 높은 성능을 보였고, Logistic Regression과 Decision Tree는 그보다 다소 낮은 성능을 보였다.
> 2차 테스트에서는 피처 셀렉션 후 전체적으로 정확도가 조금 낮아졌지만, 여전히 RandomForest가 가장 높은 성능을 유지하였다.
반면 Decision Tree는 1차보다 소폭 향상되었고, Logistic Regression은 약간 감소하였다.
이를 통해 이번 타이타닉 데이터에서는 모든 feature를 사용한 경우가 전반적으로 더 좋은 성능을 보였으며, 피처 셀렉션의 효과는 모델에 따라 다르게 나타났다고 볼 수 있다.

### 도구 설정 및 데이터 불러오기

In [4]:
import pandas as pd

# 데이터 전처리 도구
from sklearn.preprocessing import LabelEncoder

# 분류 모델
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 분류 모델 평가 도구
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split


In [5]:
path = '/Users/ganghyejun/Desktop/100_College/2-1/인공지능개론/AICLASS_2-1/task/src/data/titanic.csv' #데이터 파일 경로
df = pd.read_csv(path, index_col=0)
df

df.isnull().sum()

# pandas 라이브러리를 이용해 csv 파일을 읽어와 df에 저장
# index_col=0 : 쓸모없는 첫 번째 열을 인덱스로 빼버리니까, 실제 데이터 컬럼들이 한 칸씩 정상 위치로 정리됨

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

### 데이터 처리 1차
- 성별(Sex)이나 승선 항구(Embarked) 같은 문자열 데이터는 머신러닝 모델이 읽을 수 있도록 라벨 인코더를 사용해 숫자로 변환
- 분류 모델 3개 학습 및 평가 진행

In [6]:
y = df['Survived']
y.value_counts()

X = df.drop('Survived', axis=1)
X = X.drop(['Name', 'Ticket', 'Cabin'], axis=1)

X['Age'] = X['Age'].fillna(X['Age'].median())
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])

le = LabelEncoder()
X['Sex'] = le.fit_transform(X['Sex'])
X['Embarked'] = le.fit_transform(X['Embarked'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,test_size=0.25,random_state=0
)

### 데이터 처리 2차: 피치 셀렉션
- 각 컬럼과 생존 여부 간의 상관관계를 파악하고, 생존에 영향을 크게 미치는 핵심 특징들만 선별합니다 (피치 셀렉션)
- 피치셀렉션된 전처리 데이터로 다시 테스트 및 평가 진행

In [7]:
corr = X.copy()
corr['Survived'] = y
corr['Survived'].corr(corr['Pclass'])

corr = corr.corr(numeric_only=True)
corr['Survived'].sort_values(ascending=False)

selected_features = ['Pclass', 'Sex', 'Fare']
X2 = X[selected_features]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=0
)


In [8]:
# 모델 학습 및 평가 함수 정의
def run_model(X_train, X_test, y_train, y_test):
    clf_lr = LogisticRegression(random_state=0, max_iter=1000)
    clf_lr.fit(X_train, y_train)
    pred_lr = clf_lr.predict(X_test)
    print("\n--- Logistic Regression Classifier ---")
    print(accuracy_score(y_test, pred_lr))
    print(confusion_matrix(y_test, pred_lr))

    clf_dt = DecisionTreeClassifier(random_state=0)
    clf_dt.fit(X_train, y_train)
    pred_dt = clf_dt.predict(X_test)
    print("\n--- DecisionTreeClassifier Classifier ---")
    print(accuracy_score(y_test, pred_dt))
    print(confusion_matrix(y_test, pred_dt))

    clf_rf = RandomForestClassifier(random_state=0)
    clf_rf.fit(X_train, y_train)
    pred_rf = clf_rf.predict(X_test)
    print("\n--- RandomForest Classifier ---")
    print(accuracy_score(y_test, pred_rf))
    print(confusion_matrix(y_test, pred_rf))


In [11]:
# 테스트 시작
print("=== Test with All Features ===")
run_model(X_train, X_test, y_train, y_test)

print("\n=== Test with Selected Features ===")
run_model(X2_train, X2_test, y2_train, y2_test)

=== Test with All Features ===

--- Logistic Regression Classifier ---
0.7937219730941704
[[116  23]
 [ 23  61]]

--- DecisionTreeClassifier Classifier ---
0.7847533632286996
[[117  22]
 [ 26  58]]

--- RandomForest Classifier ---
0.8295964125560538
[[122  17]
 [ 21  63]]

=== Test with Selected Features ===

--- Logistic Regression Classifier ---
0.7802690582959642
[[115  24]
 [ 25  59]]

--- DecisionTreeClassifier Classifier ---
0.7937219730941704
[[115  24]
 [ 22  62]]

--- RandomForest Classifier ---
0.8071748878923767
[[116  23]
 [ 20  64]]
